# 00 - Arranque

Prepara el ambiente y reconstruye los datos. Ejecutar las celdas en orden, de arriba abajo,
en un servidor de Colab (CPU, RAM alta). Ver `README.md`.

In [ ]:
# PROPOSITO: dejar el codigo del repo disponible en Colab y listo para importar.
# Colab empieza vacio en cada servidor nuevo; por eso se clona desde GitHub.
# IMPORTANTE: lo que no este subido a GitHub (push) no aparece aqui.
# ESPERADO: sin errores. La primera vez muestra la clonacion; luego solo un pull.
import os
import sys

REPO = "Adriel23456/Applied-AI-Agent-Project"
BRANCH = "03DataCollectionAndEnvironmentSetup"

# Si el repo no esta, se clona; si ya esta (servidor vivo), solo se actualiza
if not os.path.exists("/content/repo"):
    !git clone -b {BRANCH} https://github.com/{REPO}.git /content/repo
else:
    !git -C /content/repo pull

# Entrar a la carpeta del repo e instalar las librerias fijadas en requirements.txt
%cd /content/repo
!pip install -q -r requirements.txt

# Permite escribir "from ptcg_wp... import ..." en cualquier celda
sys.path.insert(0, "/content/repo/src")

In [ ]:
# PROPOSITO: darle a Colab el token de Kaggle para poder descargar los datos.
# El token se lee de los secretos de Colab, de secrets.txt, o se pide a mano:
# si pide "KAGGLE_API_TOKEN:", pega el valor (no se ve mientras se escribe).
# El token NUNCA se escribe dentro del cuaderno ni se sube a GitHub.
# ESPERADO: una lista de 5 datasets de Pokemon TCG. Si sale un error 401 o 403,
# el token es incorrecto.
from ptcg_wp.colab import setup_kaggle

setup_kaggle()
!kaggle datasets list -s "pokemon tcg ai battle" | head -5

## Ingesta

Descarga cada dia de `configs/dias.yaml`, escribe dos Parquet por dia
(`partidas_<dia>` y `estados_<dia>`) y borra el crudo.

In [ ]:
# PROPOSITO: construir las tablas a partir de los dias de configs/dias.yaml.
# Por cada dia: descarga (~1.5 min), convierte los ~6500 JSON en dos tablas
# (~10.5 min) y borra los 21 GB de datos crudos.
# Si un dia ya fue procesado, se omite. Si una descarga quedo a medias, se repite sola.
# ESPERADO por dia, una linea como:
#   2026-06-18: 6532 partidas, 783197 estados | descarga 92s, procesamiento 638s
# Resultado en /content/datos: partidas_<dia>.parquet y estados_<dia>.parquet
# (se pierden al desconectar el servidor; hay que volver a correr esta celda).
!PYTHONPATH=src python -m ptcg_wp.ingest --config configs/dias.yaml --raw /content/raw --out /content/datos

In [ ]:
# PROPOSITO: confirmar que las tablas quedaron bien antes de seguir con el trabajo.
# Cada bloque indica el resultado esperado.
import pandas as pd
from pathlib import Path

d = Path("/content/datos")
games = pd.concat(pd.read_parquet(p) for p in sorted(d.glob("partidas_*.parquet")))
states = pd.concat(pd.read_parquet(p) for p in sorted(d.glob("estados_*.parquet")))

# 1) Tabla de partidas: una fila por partida, 10 columnas.
#    winner: 0 o 1 es el ganador; -1 es empate (pocas, unas 11 por dia).
print("partidas:", games.shape)
print(games["winner"].value_counts())

# 2) Tabla de estados: unas 120 filas por partida con ganador y 69 columnas.
#    Las partidas con estados deben ser las partidas menos los empates.
print("estados:", states.shape, "| partidas con estados:", states["game_id"].nunique())

# 3) Prevalencia de y: cerca de 0.54. No es 0.5 porque el ganador suele tener
#    mas pasos propios que el perdedor. Es la probabilidad a priori de la linea base B0.
print("prevalencia y:", states["y"].mean().round(3))

# 4) Filas de preparacion (turno 0, ~3.6 %) y filas de partidas anomalas
#    (esperado: muy pocas o 0). RAM de la tabla: unos 470 MB por dia.
print("filas setup:", states["setup"].sum(), "| anomalas:", states["anomalous"].sum())
print("MB en RAM:", round(states.memory_usage(deep=True).sum() / 1e6))

# 5) Los empates NO deben tener estados: "empates con estados" debe ser 0.
ids_sin_estados = set(games.game_id) - set(states.game_id)
empates_con_estados = set(games.loc[games.winner == -1, "game_id"]) & set(states.game_id)
print("partidas sin estados:", len(ids_sin_estados), "| empates con estados:", len(empates_con_estados))

# 6) En cuantas partidas el ganador tiene mas filas que el perdedor
#    (esperado: alrededor de 0.6; explica la prevalencia mayor a 0.5).
per = states.groupby(["game_id", "y"]).size().unstack(fill_value=0)
print("ganador con mas filas:", (per[1] > per[0]).mean().round(3),
      "| igual:", (per[1] == per[0]).mean().round(3))

# 7) Partidas anomalas (algun jugador no termino en DONE). Esperado: 0 o muy pocas.
print("partidas anomalas:", states.groupby("game_id")["anomalous"].max().sum())

# 8) Filas con premios en 0 fuera de la preparacion. Esperado: un numero muy pequeno
#    (son partidas terminando porque alguien tomo su ultimo premio).
fuera_setup = states[states["setup"] == 0]
print("filas con premios en 0 fuera de setup:",
      int((fuera_setup[["own_prizes", "opp_prizes"]] == 0).any(axis=1).sum()))

# 9) Distribucion del turno: mediana cerca de 6 y maximo cerca de 85.
print(states["turn"].describe())